# Cross-asset correlation

Do metals, agriculture, shipping and insurance names move with oil, and is that relationship stable? This notebook computes asset-level and sector-level return correlations, full-period correlation with oil, and rolling correlation with oil.

Prices come from Yahoo Finance via `yfinance` when `USE_YFINANCE=1`; otherwise a synthetic factor model is used so the notebook runs offline.

In [ ]:
import os
from pathlib import Path

# All inputs and outputs live under DATA_DIR (relative to the repo by default).
DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
OUT_DIR = DATA_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

USE_YFINANCE = os.environ.get("USE_YFINANCE", "0") == "1"
START_DATE = "2018-01-01"
ROLL_WINDOW = 63  # about one quarter of trading days

UNIVERSE = {
    "Energy": ["CL=F", "NG=F"],
    "Metals": ["GC=F", "SI=F", "HG=F"],
    "Agriculture": ["ZC=F", "ZW=F", "KC=F"],
    "Shipping": ["ZIM", "MATX"],
    "Insurance": ["TRV", "CB"],
}
OIL = "CL=F"
TICKER_SECTOR = {t: s for s, ts in UNIVERSE.items() for t in ts}

In [ ]:
def synthetic_prices(tickers, n_days=1500, seed=SEED):
    """One global factor, one oil factor, sector factors and idiosyncratic noise."""
    rng = np.random.default_rng(seed)
    idx = pd.bdate_range(START_DATE, periods=n_days)
    glob = rng.normal(0, 0.008, n_days)
    oil = rng.normal(0, 0.015, n_days)
    sector_f = {s: rng.normal(0, 0.008, n_days) for s in UNIVERSE}
    oil_beta = {"Energy": 1.0, "Metals": 0.3, "Agriculture": 0.25, "Shipping": 0.4, "Insurance": 0.05}
    rets = {}
    for t in tickers:
        s = TICKER_SECTOR[t]
        rets[t] = glob + oil_beta[s] * oil + sector_f[s] + rng.normal(0, 0.012, n_days)
    return 100 * np.exp(pd.DataFrame(rets, index=idx).cumsum())


def load_prices(tickers):
    if USE_YFINANCE:
        import yfinance as yf
        px = yf.download(tickers, start=START_DATE, auto_adjust=True, progress=False)["Close"]
        return px.dropna(how="all")
    return synthetic_prices(tickers)


tickers = list(TICKER_SECTOR)
prices = load_prices(tickers)
prices.tail()

In [ ]:
rets = np.log(prices).diff().dropna(how="all")
asset_corr = rets.corr()

sector_rets = rets.T.groupby(TICKER_SECTOR).mean().T  # equal-weight sector baskets
sector_corr = sector_rets.corr()

corr_with_oil = asset_corr[OIL].drop(OIL).sort_values(ascending=False)
corr_with_oil

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, m, title in [(axes[0], asset_corr, "Asset correlation"), (axes[1], sector_corr, "Sector correlation")]:
    im = ax.imshow(m.values, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(m)), m.columns, rotation=90)
    ax.set_yticks(range(len(m)), m.index)
    ax.set_title(title)
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

In [ ]:
rolling_sector = sector_rets.drop(columns="Energy").rolling(ROLL_WINDOW).corr(rets[OIL])
rolling_sector.plot(figsize=(12, 5), title=f"{ROLL_WINDOW}-day rolling correlation with oil")
plt.axhline(0, color="grey", lw=0.8)
plt.show()

# Stability check: how much does the rolling correlation move around?
rolling_sector.describe().T[["mean", "std", "min", "max"]]

In [ ]:
with pd.ExcelWriter(OUT_DIR / "cross_asset_correlation.xlsx") as xw:
    prices.to_excel(xw, sheet_name="prices")
    asset_corr.to_excel(xw, sheet_name="asset_corr")
    sector_corr.to_excel(xw, sheet_name="sector_corr")
    rolling_sector.to_excel(xw, sheet_name="rolling_vs_oil")